## Walnut Creek Streamflow Analysis
Compare recent monthly streamflow (2022-2025) against a 1991-2020
historical baseline using USGS Water Data APIs.

References:
  - Statistics API (observationIntervals): pre-computed monthly stats
    https://api.waterdata.usgs.gov/statistics/v0/docs
  - Terminology reference: https://waterdata.usgs.gov/statistics-documentation/#terms-and-descriptions-used-in-calculations
  - Get an API key at https://api.waterdata.usgs.gov/signup/

In [ ]:
import requests
import pandas as pd
import numpy as np
from io import StringIO
import os

In [ ]:
# ──────────────────────────────────────────────────────────────────────
# CONFIGURATION
# ──────────────────────────────────────────────────────────────────────

# Primary site: Walnut Creek at Des Moines, IA
# - USGS-05484800, drainage area 78.4 mi², discharge records since Oct 1971
SITE_ID = "USGS-05484800"

# Backup sites (uncomment to switch):
# SITE_ID = "USGS-05484700"   # Walnut Creek at West Des Moines, IA (64 mi², records since 1957)
# SITE_ID = "USGS-05484500"   # Raccoon River at Van Meter, IA (3,441 mi², records since 1915)
# NOTE: Do NOT use USGS-05484600 (Raccoon River near West Des Moines) —
#       that site is stage-only with discharge published only during low flow.

PARAM_CODE = "00060"  # Discharge, cubic feet per second

# Date ranges
BASELINE_START = "1991-01-01"
BASELINE_END = "2020-12-31"
RECENT_START = "2022-01-01"
RECENT_END = "2025-12-31"

STATS_BASE = "https://api.waterdata.usgs.gov/statistics/v0"

In [ ]:
# ──────────────────────────────────────────────────────────────────────
# HELPER: build common request params
# ──────────────────────────────────────────────────────────────────────


def _add_api_key(params: dict) -> dict:
    usgs_api_key = os.getenv("USGS_API_KEY")
    if usgs_api_key:
        params["api_key"] = usgs_api_key
    else:
        raise ValueError(
            "USGS API key not found. Set it as an environment variable USGS_API_KEY."
        )
    return params

In [ ]:
def unnest_rows(df: pd.DataFrame, nested_col: str) -> pd.DataFrame:
    """
    Unnest a column containing lists of records into separate rows.

    For each row in the input DataFrame, if the specified nested_col contains
    a list of records, this function will create a new row for each record,
    combining it with the other columns from the original row.

    Args:
        df: Input DataFrame with a nested column.
        nested_col: Name of the column that contains lists of records to unnest.

    Returns:
        A new DataFrame with unnested rows.
    """
    records = []
    for _, row in df.iterrows():
        nested = row[nested_col]
        if isinstance(nested, list):
            meta = row.drop(nested_col).to_dict()
            for rec in nested:
                records.append({**meta, **rec})
    return pd.DataFrame(records) if records else pd.DataFrame()

In [ ]:
# ──────────────────────────────────────────────────────────────────────
# Historical baseline (1991–2020) via observationIntervals
# ──────────────────────────────────────────────────────────────────────
#
# The observationIntervals endpoint returns calendar-month, calendar-year,
# and water-year statistics for a specified date range.  Each calendar-month
# record summarises the daily mean values within that specific month/year
# (e.g. "January 1991", "January 1992", …, "January 2020").
#
# We request all available computation types so we get mean, median, min,
# max, and percentiles per month.  We then group by calendar month (1–12)
# and compute the cross-year median (P50) and 75th percentile (P75) of
# those monthly values.
#
# The endpoint only uses *approved* data; provisional data is excluded.
# ──────────────────────────────────────────────────────────────────────


def fetch_observation_intervals(
    site_id: str, start_date: str, end_date: str
) -> pd.DataFrame:
    """
    Fetch calendar-month statistics from the observationIntervals endpoint.

    Returns a DataFrame with one row per interval record, with columns including:
        monitoring_location_id, interval_type, start_date, end_date,
        value, computation, sample_count, approval_status, ...
    """
    url = f"{STATS_BASE}/observationIntervals"
    params = _add_api_key(
        {
            "monitoring_location_id": site_id,
            "parameter_code": PARAM_CODE,
            "start_date": start_date,
            "end_date": end_date,
        }
    )

    print(f"Requesting observationIntervals: {url}")
    print(f"  params: {params}")

    resp = requests.get(url, params=params, timeout=120)
    resp.raise_for_status()

    data = resp.json()

    if isinstance(data, list):
        df = pd.DataFrame(data)
    elif isinstance(data, dict) and "results" in data:
        df = pd.DataFrame(data["results"])
    elif isinstance(data, dict) and "features" in data:
        records = [f["properties"] for f in data["features"]]
        df = pd.DataFrame(records)
    else:
        print(
            f"  Unexpected response structure. Top-level keys: {list(data.keys()) if isinstance(data, dict) else type(data)}"
        )
        df = pd.DataFrame()

    # Unnest the 'data' column (list of interval records) and then 'values' column (list of computed stats)
    df = unnest_rows(df, "data")
    df = unnest_rows(df, "values")

    print(f"  Retrieved {len(df)} records")
    if not df.empty:
        print(f"  Columns: {list(df.columns)}")
        print(f"  Sample row:\n{df.iloc[0].to_dict()}")

    return df

In [ ]:
print("\nFetching baseline period from observationIntervals...")
baseline_raw = fetch_observation_intervals(SITE_ID, BASELINE_START, BASELINE_END)

if baseline_raw.empty:
    raise ValueError("Empty response from observationIntervals")

In [ ]:
print("\nFetching recent period from observationIntervals...")
recent_raw = fetch_observation_intervals(SITE_ID, RECENT_START, RECENT_END)

if recent_raw.empty:
    raise ValueError("Empty response from observationIntervals")

In [ ]:
def compute_baseline_thresholds(df: pd.DataFrame) -> pd.DataFrame:
    """
    From the observationIntervals output (one row per interval record),
    filter to calendar-month / arithmetic_mean rows, extract the month number,
    then compute P5, P10, P25, and P50 for each calendar month 1–12.
    """
    # Filter to monthly interval records only (exclude calendar_year, water_year)
    if "interval_type" in df.columns:
        monthly = df[
            df["interval_type"].str.contains("month", case=False, na=False)
        ].copy()
    else:
        print(
            f"  WARNING: 'interval_type' column not found. Available: {list(df.columns)}"
        )
        monthly = df.copy()

    # If multiple computation types are present (mean, median, …), keep arithmetic_mean
    if "computation" in monthly.columns:
        mean_rows = monthly[monthly["computation"] == "arithmetic_mean"]
        if not mean_rows.empty:
            monthly = mean_rows

    # Extract calendar month from start_date
    date_col = next(
        (c for c in ["start_date", "begin_date"] if c in monthly.columns), None
    )
    if date_col:
        monthly["month"] = pd.to_datetime(monthly[date_col]).dt.month
    elif "month" in monthly.columns:
        pass
    else:
        print(
            f"  WARNING: Could not find date column. Available: {list(monthly.columns)}"
        )
        return pd.DataFrame()

    # Identify the value column
    value_col = next(
        (
            c
            for c in ["value", "median", "arithmetic_mean", "mean"]
            if c in monthly.columns
        ),
        None,
    )
    if value_col is None:
        print(
            f"  WARNING: Could not find value column. Available: {list(monthly.columns)}"
        )
        return pd.DataFrame()

    monthly[value_col] = pd.to_numeric(monthly[value_col], errors="coerce")

    baseline = (
        monthly.groupby("month")[value_col]
        .agg(
            baseline_p5=lambda x: np.nanpercentile(x, 5),
            baseline_p10=lambda x: np.nanpercentile(x, 10),
            baseline_p25=lambda x: np.nanpercentile(x, 25),
            baseline_p50=lambda x: np.nanpercentile(x, 50),
            n_years="count",
        )
        .reset_index()
    )

    print(f"  Computed baseline thresholds for {len(baseline)} months")
    return baseline

In [ ]:
baseline = compute_baseline_thresholds(baseline_raw)

if baseline.empty:
    raise ValueError("Could not compute baseline from observationIntervals")

print("\nBaseline thresholds:")
print(baseline)

In [ ]:
# ──────────────────────────────────────────────────────────────────────
# Recent period (2022–2025) via observationIntervals
# ──────────────────────────────────────────────────────────────────────


def extract_recent_monthly(df: pd.DataFrame) -> pd.DataFrame:
    """
    From the observationIntervals output for the recent period,
    filter to calendar-month / arithmetic_mean rows and extract
    month, year, and the monthly summary value.
    """
    if "interval_type" in df.columns:
        monthly = df[
            df["interval_type"].str.contains("month", case=False, na=False)
        ].copy()
    else:
        monthly = df.copy()

    if "computation" in monthly.columns:
        mean_rows = monthly[monthly["computation"] == "arithmetic_mean"]
        if not mean_rows.empty:
            monthly = mean_rows

    date_col = next(
        (c for c in ["start_date", "begin_date"] if c in monthly.columns), None
    )
    if date_col:
        monthly["date"] = pd.to_datetime(monthly[date_col])
        monthly["year"] = monthly["date"].dt.year
        monthly["month"] = monthly["date"].dt.month
    elif "month" in monthly.columns and "year" in monthly.columns:
        pass
    else:
        print(f"  WARNING: Could not parse dates. Columns: {list(monthly.columns)}")
        return pd.DataFrame()

    value_col = next(
        (
            c
            for c in ["value", "median", "arithmetic_mean", "mean"]
            if c in monthly.columns
        ),
        None,
    )
    if value_col:
        monthly[value_col] = pd.to_numeric(monthly[value_col], errors="coerce")
        monthly = monthly.rename(columns={value_col: "monthly_value"})

    return monthly

In [ ]:
recent = extract_recent_monthly(recent_raw)

if recent.empty:
    raise ValueError("Could not extract recent monthly data from observationIntervals")

print("\nRecent monthly values:")
print(recent[["year", "month", "monthly_value"]])

In [ ]:
def join_recent_to_baseline(
    recent: pd.DataFrame, baseline: pd.DataFrame
) -> pd.DataFrame:
    """
    Merge the recent monthly values with baseline P5, P10, P25, and P50
    thresholds by calendar month.
    """
    merged = recent.merge(baseline, on="month", how="left")

    # Flag months where recent flow falls below the baseline P25
    if "monthly_value" in merged.columns and "baseline_p25" in merged.columns:
        merged["below_p25"] = merged["monthly_value"] < merged["baseline_p25"]

    return merged

In [ ]:
print("\nJoining recent data to baseline thresholds...")
result = join_recent_to_baseline(recent, baseline)

print("\n" + "=" * 70)
print("BASELINE THRESHOLDS (1991–2020) by calendar month")
print("  P5/P10/P25 = low-flow drought thresholds")
print("  P50        = median")
print("=" * 70)
month_names = {
    1: "Jan",
    2: "Feb",
    3: "Mar",
    4: "Apr",
    5: "May",
    6: "Jun",
    7: "Jul",
    8: "Aug",
    9: "Sep",
    10: "Oct",
    11: "Nov",
    12: "Dec",
}
baseline["month_name"] = baseline["month"].map(month_names)
print(
    baseline[
        [
            "month",
            "month_name",
            "baseline_p5",
            "baseline_p10",
            "baseline_p25",
            "baseline_p50",
            "n_years",
        ]
    ].to_string(index=False)
)

print("\n" + "=" * 70)
print("RECENT MONTHLY VALUES (2022–2025) vs BASELINE")
print("=" * 70)
if not result.empty:
    if "year" in result.columns and "month" in result.columns:
        result = result.sort_values(["year", "month"])
    cols = [
        c
        for c in [
            "year",
            "month",
            "monthly_value",
            "baseline_p25",
            "baseline_p50",
            "below_p25",
        ]
        if c in result.columns
    ]
    print(result[cols].to_string(index=False))
else:
    print("  No recent data available (may still be provisional).")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

MONTH_LABELS = [
    "Jan",
    "Feb",
    "Mar",
    "Apr",
    "May",
    "Jun",
    "Jul",
    "Aug",
    "Sep",
    "Oct",
    "Nov",
    "Dec",
]

# Colors shared between bars and their corresponding threshold lines
COLOR_P5 = "#c0392b"  # red
COLOR_P10 = "#8e44ad"  # purple
COLOR_P25 = "#e67e22"  # orange
COLOR_P50 = "#2980b9"  # blue
COLOR_ABOVE = "#27ae60"  # green (above P50)


def bar_color(flow, p5, p10, p25, p50):
    """Return the color of the lowest percentile threshold the bar falls below."""
    if pd.isna(flow):
        return COLOR_P50
    if flow < p5:
        return COLOR_P5
    if flow < p10:
        return COLOR_P10
    if flow < p25:
        return COLOR_P25
    if flow < p50:
        return COLOR_P50
    return COLOR_ABOVE


years = sorted(result["year"].dropna().astype(int).unique())
fig, axes = plt.subplots(len(years), 1, figsize=(13, 4 * len(years)), sharex=False)
if len(years) == 1:
    axes = [axes]

for ax, year in zip(axes, years):
    yr = result[result["year"] == year].sort_values("month")
    if yr.empty:
        ax.set_visible(False)
        continue

    x = list(range(len(yr)))
    colors = [
        bar_color(
            row["monthly_value"],
            row["baseline_p5"],
            row["baseline_p10"],
            row["baseline_p25"],
            row["baseline_p50"],
        )
        for _, row in yr.iterrows()
    ]

    ax.bar(x, yr["monthly_value"], color=colors, alpha=0.85, zorder=2)
    ax.plot(
        x,
        yr["baseline_p50"],
        color=COLOR_P50,
        linestyle="--",
        linewidth=1.8,
        marker="o",
        markersize=4,
        zorder=3,
        label="Baseline P50",
    )
    ax.plot(
        x,
        yr["baseline_p25"],
        color=COLOR_P25,
        linestyle="--",
        linewidth=1.8,
        marker="s",
        markersize=4,
        zorder=3,
        label="Baseline P25",
    )
    ax.plot(
        x,
        yr["baseline_p10"],
        color=COLOR_P10,
        linestyle=":",
        linewidth=1.5,
        marker="^",
        markersize=4,
        zorder=3,
        label="Baseline P10",
    )
    ax.plot(
        x,
        yr["baseline_p5"],
        color=COLOR_P5,
        linestyle=":",
        linewidth=1.5,
        marker="v",
        markersize=4,
        zorder=3,
        label="Baseline P5",
    )

    ax.set_xticks(x)
    ax.set_xticklabels([MONTH_LABELS[int(m) - 1] for m in yr["month"]])
    ax.set_title(str(year), fontsize=13, fontweight="bold", loc="left")
    ax.set_ylabel("Discharge (ft³/s)")
    ax.yaxis.grid(True, linestyle=":", alpha=0.5)
    ax.set_axisbelow(True)

    if year == years[0]:
        bar_legend = [
            mpatches.Patch(color=COLOR_ABOVE, alpha=0.85, label="Flow ≥ P50"),
            mpatches.Patch(color=COLOR_P50, alpha=0.85, label="P25 ≤ flow < P50"),
            mpatches.Patch(color=COLOR_P25, alpha=0.85, label="P10 ≤ flow < P25"),
            mpatches.Patch(color=COLOR_P10, alpha=0.85, label="P5 ≤ flow < P10"),
            mpatches.Patch(color=COLOR_P5, alpha=0.85, label="Flow < P5"),
        ]
        line_handles, _ = ax.get_legend_handles_labels()
        ax.legend(handles=bar_legend + line_handles, fontsize=9, loc="upper right")

fig.suptitle(
    "Monthly Streamflow vs. 1991–2020 Historical Baseline\n"
    "Walnut Creek at Des Moines, IA  (USGS-05484800, parameter 00060)",
    fontsize=13,
    y=1.01,
)
plt.tight_layout()
plt.show()

## Annual Analysis (calendar_year interval type)

Compare each recent calendar year's mean discharge against percentile thresholds
derived from the full 1991–2020 baseline distribution of annual means.

In [ ]:
# ──────────────────────────────────────────────────────────────────────
# Annual baseline thresholds and recent annual values
# ──────────────────────────────────────────────────────────────────────


def compute_annual_baseline_thresholds(df: pd.DataFrame) -> dict:
    """
    Filter to calendar_year / arithmetic_mean records and compute
    P5, P10, P25, P50 across all baseline years.
    """
    annual = df[
        df["interval_type"].str.contains("calendar_year", case=False, na=False)
    ].copy()

    if "computation" in annual.columns:
        mean_rows = annual[annual["computation"] == "arithmetic_mean"]
        if not mean_rows.empty:
            annual = mean_rows

    value_col = next(
        (
            c
            for c in ["value", "median", "arithmetic_mean", "mean"]
            if c in annual.columns
        ),
        None,
    )
    if value_col is None:
        print(
            f"  WARNING: Could not find value column. Available: {list(annual.columns)}"
        )
        return {}

    vals = pd.to_numeric(annual[value_col], errors="coerce").dropna()
    thresholds = {
        "baseline_p5": float(np.nanpercentile(vals, 5)),
        "baseline_p10": float(np.nanpercentile(vals, 10)),
        "baseline_p25": float(np.nanpercentile(vals, 25)),
        "baseline_p50": float(np.nanpercentile(vals, 50)),
        "n_years": int(len(vals)),
    }
    print(f"  Computed annual thresholds from {thresholds['n_years']} years")
    return thresholds


def extract_recent_annual(df: pd.DataFrame) -> pd.DataFrame:
    """
    Filter to calendar_year / arithmetic_mean records for the recent period
    and return a DataFrame with year and annual_value columns.
    """
    annual = df[
        df["interval_type"].str.contains("calendar_year", case=False, na=False)
    ].copy()

    if "computation" in annual.columns:
        mean_rows = annual[annual["computation"] == "arithmetic_mean"]
        if not mean_rows.empty:
            annual = mean_rows

    date_col = next(
        (c for c in ["start_date", "begin_date"] if c in annual.columns), None
    )
    if date_col:
        annual["year"] = pd.to_datetime(annual[date_col]).dt.year

    value_col = next(
        (
            c
            for c in ["value", "median", "arithmetic_mean", "mean"]
            if c in annual.columns
        ),
        None,
    )
    if value_col:
        annual[value_col] = pd.to_numeric(annual[value_col], errors="coerce")
        annual = annual.rename(columns={value_col: "annual_value"})

    return annual[["year", "annual_value"]].sort_values("year").reset_index(drop=True)


print("\nComputing annual baseline thresholds (1991–2020)...")
annual_baseline = compute_annual_baseline_thresholds(baseline_raw)

print("\nExtracting recent annual values (2022–2025)...")
recent_annual = extract_recent_annual(recent_raw)
for k, v in annual_baseline.items():
    recent_annual[k] = v

In [ ]:
print("\n" + "=" * 70)
print("ANNUAL BASELINE THRESHOLDS (1991–2020)")
print("=" * 70)
for label, key in [
    ("P5", "baseline_p5"),
    ("P10", "baseline_p10"),
    ("P25", "baseline_p25"),
    ("P50", "baseline_p50"),
]:
    print(f"  {label}: {annual_baseline[key]:.1f} ft\u00b3/s")
print(f"  n_years: {annual_baseline['n_years']}")

print("\n" + "=" * 70)
print("RECENT ANNUAL VALUES (2022–2025) vs BASELINE")
print("=" * 70)
cols = [
    c
    for c in ["year", "annual_value", "baseline_p25", "baseline_p50"]
    if c in recent_annual.columns
]
print(recent_annual[cols].to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

a_colors = [
    bar_color(
        row["annual_value"],
        row["baseline_p5"],
        row["baseline_p10"],
        row["baseline_p25"],
        row["baseline_p50"],
    )
    for _, row in recent_annual.iterrows()
]

ax.bar(
    recent_annual["year"],
    recent_annual["annual_value"],
    color=a_colors,
    alpha=0.85,
    width=0.6,
    zorder=2,
)

ax.axhline(
    annual_baseline["baseline_p50"],
    color=COLOR_P50,
    linestyle="--",
    linewidth=1.8,
    label="Baseline P50",
)
ax.axhline(
    annual_baseline["baseline_p25"],
    color=COLOR_P25,
    linestyle="--",
    linewidth=1.8,
    label="Baseline P25",
)
ax.axhline(
    annual_baseline["baseline_p10"],
    color=COLOR_P10,
    linestyle=":",
    linewidth=1.5,
    label="Baseline P10",
)
ax.axhline(
    annual_baseline["baseline_p5"],
    color=COLOR_P5,
    linestyle=":",
    linewidth=1.5,
    label="Baseline P5",
)

ax.set_xticks(recent_annual["year"])
ax.set_xticklabels(recent_annual["year"].astype(int))
ax.set_ylabel("Mean Annual Discharge (ft\u00b3/s)")
ax.set_title(
    "Annual Mean Streamflow vs. 1991\u20132020 Historical Baseline\n"
    "Walnut Creek at Des Moines, IA  (USGS-05484800, parameter 00060)",
    fontsize=12,
)
ax.yaxis.grid(True, linestyle=":", alpha=0.5)
ax.set_axisbelow(True)

bar_legend = [
    mpatches.Patch(color=COLOR_ABOVE, alpha=0.85, label="Flow \u2265 P50"),
    mpatches.Patch(color=COLOR_P50, alpha=0.85, label="P25 \u2264 flow < P50"),
    mpatches.Patch(color=COLOR_P25, alpha=0.85, label="P10 \u2264 flow < P25"),
    mpatches.Patch(color=COLOR_P10, alpha=0.85, label="P5 \u2264 flow < P10"),
    mpatches.Patch(color=COLOR_P5, alpha=0.85, label="Flow < P5"),
]
line_handles, _ = ax.get_legend_handles_labels()
ax.legend(handles=bar_legend + line_handles, fontsize=9, loc="upper right")

plt.tight_layout()
plt.show()

In [ ]:
# ------------------------------------------------------------------
# Save to CSV
# ------------------------------------------------------------------
baseline_file = "../output_files/baseline_thresholds_1991_2020.csv"
result_file = "../output_files/recent_vs_baseline_2022_2025.csv"
annual_baseline_file = "../output_files/annual_baseline_thresholds_1991_2020.csv"
annual_result_file = "../output_files/annual_recent_vs_baseline_2022_2025.csv"

baseline.to_csv(baseline_file, index=False)
print(f"\nSaved monthly baseline thresholds to: {baseline_file}")

if not result.empty:
    result.to_csv(result_file, index=False)
    print(f"Saved monthly comparison table to:    {result_file}")

pd.DataFrame([annual_baseline]).to_csv(annual_baseline_file, index=False)
print(f"Saved annual baseline thresholds to:  {annual_baseline_file}")

if not recent_annual.empty:
    recent_annual.to_csv(annual_result_file, index=False)
    print(f"Saved annual comparison table to:     {annual_result_file}")

print("\nDone.")